In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
#dbutils.widgets.text('table_name','')
table_name=dbutils.widgets.get('table_name')

In [0]:
meta_df=spark.sql(f"""
                  select source_table,target_table,key_col,cond_cols from my_catalog.bronze.etl_control where table_name='{table_name}'
                  """)

row=meta_df.collect()[0]
source_table=row['source_table']
target_table=row['target_table']
key_col=row['key_col']
cond_cols=row['cond_cols']

condition="OR".join([f"src.{col}<>tgt.{col}" for col in cond_cols.split(",")])

In [0]:
source_path='/Volumes/realtimeprojects/eteproject/scd/Employee_2.csv'

df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(source_path)


In [0]:
#df.write.saveAsTable('my_catalog.silver.emp_src')
target_table1="my_catalog.silver.emp_src"
df.write.format("delta").mode("overwrite").saveAsTable(target_table1)

In [0]:
%sql
select * from my_catalog.silver.emp_src

In [0]:
df=spark.sql("""
select * from my_catalog.silver.emp_src
""")

In [0]:
df=spark.sql("""
select a.*,to_date(current_timestamp(),'yyyy-MM-dd') as START_DATE,to_date('9999-12-31','yyyy-MM-dd') as END_DATE,1 as IS_ACTIVE from my_catalog.silver.emp_src a
""")

In [0]:
df.createOrReplaceTempView("srctemp")

In [0]:
#mark inactive
spark.sql(f"""
merge into {target_table} as tgt
using {source_table} as src
on src.{key_col}=tgt.{key_col} and tgt.is_active=1
when matched then update set tgt.end_date=to_date(current_timestamp(),'yyyy-MM-dd'),
tgt.is_active=0
""")

In [0]:
#insert+updated records
spark.sql(f"""
merge into {target_table} as tgt
using {source_table} as b
on b.{key_col}=tgt.{key_col} and tgt.is_active=1
WHEN NOT MATCHED THEN Insert *
 """)

In [0]:
%sql
update my_catalog.bronze.etl_control set LAST_MODIFIED=current_timestamp() where table_name='{target_table}'
and active=1;

In [0]:
%sql
select * from my_catalog.gold.DIM_EMPLOYEE order by emp_id;

In [0]:
%sql
select * from my_catalog.bronze.etl_control;